# NLP on Steroids 💉 — NLP Klasik di GPU dengan NVIDIA RAPIDS

**Modul 03 · Notebook 02 · Bootcamp NCA-GENL**

Di notebook 01 kita belajar alur NLP klasik: *preprocessing → vektorisasi → klasifikasi*.
Di notebook ini kita jalankan alur yang **sama persis** — tapi di **GPU**, dengan stack **NVIDIA RAPIDS**:

| Library | Peran |
|---|---|
| **cuDF** | pandas versi GPU (DataFrame) |
| **nvtext** | operasi teks GPU di dalam cuDF (tokenize, n-gram, minhash) |
| **cuML** | scikit-learn versi GPU (TF-IDF, Logistic Regression, dll.) |

> ⚠️ **Wajib GPU**: Runtime → Change runtime type → **T4 GPU**, baru jalankan sel di bawah.

**Yang akan kamu kuasai:**
1. Menjalankan kode pandas/sklearn yang sudah kamu tulis di GPU **tanpa mengubah satu baris pun**
2. Preprocessing ±150 ribu paragraf Wikipedia dalam hitungan detik
3. Deduplikasi *near-duplicate* dengan MinHash — langkah nyata penyiapan korpus LLM
4. TF-IDF + klasifikasi sentimen dengan API native cuML

In [ ]:
# Cek GPU — harus muncul Tesla T4
!nvidia-smi

In [ ]:
# RAPIDS sudah pre-installed di runtime GPU Colab — tinggal import.
# Kalau sel ini error, jalankan sel fallback di bawahnya.
import cudf, cuml
print("cuDF :", cudf.__version__)
print("cuML :", cuml.__version__)

In [ ]:
# FALLBACK — jalankan sel ini HANYA jika import di atas gagal (butuh ±5 menit).
# Hapus tanda pagar di baris bawah, lalu jalankan:
# !pip install -q --extra-index-url=https://pypi.nvidia.com cudf-cu12 cuml-cu12
print("Jika import cudf/cuml di atas BERHASIL, lewati sel ini.")

## Babak 1 — Sulap: Kode Lama, GPU Baru 🎩

Klaim NVIDIA: kode pandas + sklearn yang sudah kamu tulis bisa langsung jalan di GPU, **nol perubahan kode**, lewat dua "akselerator":

- `cudf.pandas` — mengambil alih `import pandas`, semua operasi dialihkan ke cuDF (GPU); yang tidak didukung otomatis *fallback* ke pandas asli.
- `cuml.accel` — hal yang sama untuk estimator scikit-learn → cuML.

**Caranya bukan dengan mengubah kode, tapi dengan mengubah cara menjalankannya.** Karena akselerator harus aktif *sebelum* `import pandas`, kita tulis pipeline sebagai script lalu jalankan dua kali: normal (CPU) dan lewat akselerator (GPU).

Data: **SmSA** dari benchmark IndoNLU — ulasan berbahasa Indonesia berlabel sentimen (positive/negative/neutral), dataset yang sama yang disebut di notebook 01.

In [ ]:
# Download SmSA (URL di-pin ke commit agar stabil)
!wget -q "https://raw.githubusercontent.com/IndoNLP/indonlu/ce728f6926a36174b9923dfe49d6a6839b6e9bb7/dataset/smsa_doc-sentiment-prosa/train_preprocess.tsv" -O smsa_train.tsv
!wget -q "https://raw.githubusercontent.com/IndoNLP/indonlu/ce728f6926a36174b9923dfe49d6a6839b6e9bb7/dataset/smsa_doc-sentiment-prosa/valid_preprocess.tsv" -O smsa_valid.tsv
!wc -l smsa_train.tsv smsa_valid.tsv

In [ ]:
%%writefile pipeline.py
# Pipeline klasik notebook 01: TF-IDF + Logistic Regression.
# TIDAK ADA satu baris pun kode GPU di file ini.
import time
t_all = time.perf_counter()
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

train = pd.read_csv("smsa_train.tsv", sep="\t", names=["text", "label"])
valid = pd.read_csv("smsa_valid.tsv", sep="\t", names=["text", "label"])
# Gandakan data train 20x: simulasi beban skala produksi (~220 ribu dokumen)
train = pd.concat([train] * 20, ignore_index=True)

t0 = time.perf_counter()
vec = TfidfVectorizer(max_features=50_000, ngram_range=(1, 2))
Xtr = vec.fit_transform(train.text)
Xva = vec.transform(valid.text)
t_vec = time.perf_counter() - t0

t0 = time.perf_counter()
clf = LogisticRegression(max_iter=200)
clf.fit(Xtr, train.label)
acc = accuracy_score(valid.label, clf.predict(Xva))
t_fit = time.perf_counter() - t0
print(f"akurasi={acc:.4f}  vectorize={t_vec:.1f}s  fit+predict={t_fit:.1f}s  total={time.perf_counter()-t_all:.1f}s")

In [ ]:
# Baseline: CPU murni (pandas + sklearn biasa)
!python pipeline.py

In [ ]:
# Sulapnya: file yang SAMA, dijalankan lewat akselerator GPU
!python -m cudf.pandas -m cuml.accel pipeline.py

### Apa yang barusan terjadi?

- **Akurasi sama** → hasil komputasinya identik secara statistik.
- **Waktu beda jauh** → bagian pandas (load, concat, string) jalan di cuDF, estimator sklearn yang didukung jalan di cuML.
- Operasi yang belum didukung GPU otomatis **fallback ke CPU** — kode tidak pernah crash karena ini.

> 💡 **Kapan trik ini menguntungkan?** Saat datanya cukup besar. Untuk data kecil, waktu transfer CPU↔GPU lebih mahal daripada komputasinya sendiri — GPU justru bisa lebih lambat. Ini pola umum di semua komputasi GPU.

## Babak 2 — Bedah nvtext: Preprocessing Skala Wikipedia 🔬

Sulap selesai; sekarang kita pakai API GPU-nya **secara eksplisit**. cuDF punya modul teks (*nvtext*) dengan operasi yang **sama sekali tidak ada** di pandas: `tokenize`, `token_count`, `ngrams_tokenize`, `minhash`.

Korpusnya pun naik kelas: **Wikipedia Bahasa Indonesia** — kita ambil ±150 ribu paragraf via streaming (tanpa download dump penuh).

In [ ]:
!pip install -q datasets

from datasets import load_dataset

# Streaming: ambil 20 ribu artikel pertama, pecah jadi paragraf
ds = load_dataset("wikimedia/wikipedia", "20231101.id", split="train", streaming=True)
paragraphs = []
for i, art in enumerate(ds):
    if i >= 20_000:
        break
    paragraphs += [p for p in art["text"].split("\n") if len(p) > 80]

print(f"{len(paragraphs):,} paragraf terkumpul")

In [ ]:
import time
import pandas as pd
import cudf

pdf = pd.Series(paragraphs, name="text")     # CPU
gdf = cudf.Series(paragraphs, name="text")   # GPU

BENCH = {}  # hasil benchmark, dipakai untuk ringkasan & slide

def bench(group, label, fn):
    t0 = time.perf_counter()
    out = fn()
    dt = time.perf_counter() - t0
    BENCH.setdefault(group, {})[label] = dt
    print(f"{group:12s} | {label:6s} | {dt:7.3f} s")
    return out

In [ ]:
# --- Tokenisasi + hitung token ---
n_cpu = bench("tokenize", "CPU", lambda: pdf.str.split().map(len).sum())
n_gpu = bench("tokenize", "GPU", lambda: gdf.str.token_count().sum())
print(f"total token: CPU={n_cpu:,} GPU={n_gpu:,}")

In [ ]:
# --- Frekuensi kata (top-15) ---
top_cpu = bench("wordcount", "CPU", lambda: pdf.str.lower().str.split().explode().value_counts().head(15))
top_gpu = bench("wordcount", "GPU", lambda: gdf.str.lower().str.tokenize().value_counts().head(15))
print(top_gpu)

In [ ]:
# --- Bigram (n-gram dengan n=2) ---
def bigram_cpu():
    toks = pdf.str.lower().str.split()
    return toks.map(lambda t: ["_".join(p) for p in zip(t, t[1:])]).explode().value_counts().head(10)

def bigram_gpu():
    return gdf.str.lower().str.ngrams_tokenize(2, separator="_").value_counts().head(10)

bench("bigram", "CPU", bigram_cpu)
big = bench("bigram", "GPU", bigram_gpu)
print(big)

### Skor sementara

Jalankan sel di bawah untuk rekap. Pola yang perlu kamu perhatikan: makin intensif operasi string-nya (tokenize, n-gram), makin besar kemenangan GPU — karena ribuan core mengerjakan ribuan string sekaligus.

In [ ]:
import json

print(f"{'operasi':12s} {'CPU (s)':>9s} {'GPU (s)':>9s} {'speedup':>9s}")
for op, r in BENCH.items():
    if "CPU" in r and "GPU" in r:
        print(f"{op:12s} {r['CPU']:9.3f} {r['GPU']:9.3f} {r['CPU']/r['GPU']:8.1f}x")

# Simpan untuk figure slide (instruktur: download file ini setelah smoke-test)
with open("benchmark_results.json", "w") as f:
    json.dump(BENCH, f, indent=2)

## Babak 3 — Berburu Kembar: Dedup dengan MinHash 👯

Korpus mentah penuh duplikat dan *near-duplicate* (artikel template, boilerplate, copy-paste). Untuk training LLM, duplikat = model menghafal + komputasi terbuang. Masalahnya: membandingkan semua pasangan dari 150 ribu paragraf = ±11 **miliar** perbandingan. Tidak mungkin.

**MinHash** mengambil jalan pintas: setiap dokumen diringkas jadi *sidik jari* kecil; dokumen yang mirip menghasilkan sidik jari yang sama dengan probabilitas tinggi. Kita cukup mengelompokkan sidik jari yang identik — dari miliaran perbandingan jadi satu `groupby`.

> Catatan: kita memakai *satu* sidik jari utuh, jadi hanya duplikat yang nyaris identik yang tertangkap. Sistem produksi (MinHash-LSH) memakai banyak *band* sidik jari agar kemiripan parsial pun terdeteksi.

In [ ]:
import cudf

# API minhash cuDF sedikit berbeda antar versi — helper ini mencoba keduanya.
def minhash_signature(series, n_hashes=8, width=5):
    try:  # cuDF >= 24.12
        import cupy as cp
        a = cudf.Series(cp.random.RandomState(42).randint(1, 2**31, n_hashes), dtype="uint32")
        b = cudf.Series(cp.random.RandomState(7).randint(0, 2**31, n_hashes), dtype="uint32")
        return series.str.minhash(0, a=a, b=b, width=width)
    except TypeError:  # API lama: minhash(seeds, width)
        seeds = cudf.Series(range(n_hashes), dtype="uint32")
        return series.str.minhash(seeds, width=width)

sig = minhash_signature(gdf.str.lower())
# Sidik jari = gabungan seluruh nilai minhash jadi satu string kunci
fingerprint = sig.list.astype("str").str.join("-")
df = cudf.DataFrame({"text": gdf, "fp": fingerprint})
dup_counts = df.groupby("fp").size()
dup_groups = dup_counts[dup_counts > 1]
print(f"{len(dup_groups):,} kelompok near-duplicate ditemukan")

In [ ]:
# Intip kelompok kembar terbesar
biggest_fp = dup_groups.sort_values(ascending=False).index[0]
twins = df[df.fp == biggest_fp].text.to_pandas()
for t in twins.head(3):
    print("—", t[:160], "...")

> 🧠 Yang barusan kamu lakukan — minhash + groupby di GPU — adalah versi mini dari **fuzzy deduplication** yang dijalankan **NVIDIA NeMo Curator** saat menyiapkan korpus training LLM. Bedanya cuma skala: kita memproses 150 ribu paragraf di 1 GPU, Curator memproses miliaran dokumen di ratusan GPU.

## Babak 4 — cuML Native: TF-IDF + Sentimen, Tanpa Sulap 🛠️

Di Babak 1 GPU bekerja diam-diam di balik `cuml.accel`. Sekarang kita panggil cuML **langsung** — perhatikan API-nya sengaja dibuat serupa dengan scikit-learn, jadi yang kamu pelajari di notebook 01 dipakai lagi 1:1. Tapi sebelum mengukur apa pun, ada satu ritual wajib benchmark GPU: *warm-up*.

In [ ]:
import cudf
from cuml.feature_extraction.text import TfidfVectorizer as GpuTfidf
from cuml.linear_model import LogisticRegression as GpuLogReg

train_g = cudf.read_csv("smsa_train.tsv", sep="\t", names=["text", "label"])
valid_g = cudf.read_csv("smsa_valid.tsv", sep="\t", names=["text", "label"])
LABEL2ID = {"negative": 0.0, "neutral": 1.0, "positive": 2.0}
ytr = train_g.label.map(LABEL2ID).astype("float32")
yva = valid_g.label.map(LABEL2ID).astype("float32")

# Warm-up: pemanggilan CUDA pertama membayar inisialisasi context + kompilasi
# kernel (bisa beberapa detik) — itu BUKAN kecepatan komputasi sesungguhnya.
# Aturan emas benchmark GPU: jangan pernah mengukur pemanggilan pertama.
_w = GpuTfidf(max_features=500).fit_transform(train_g.text.head(256))
_y = cudf.Series([float(i % 2) for i in range(256)], dtype="float32")
GpuLogReg(max_iter=50).fit(_w, _y)
print("GPU sudah panas — pengukuran berikutnya adil")

In [ ]:
import time

t0 = time.perf_counter()
gvec = GpuTfidf(max_features=20_000)
Xtr = gvec.fit_transform(train_g.text)
Xva = gvec.transform(valid_g.text)
gclf = GpuLogReg(max_iter=200)
gclf.fit(Xtr, ytr)
from cuml.metrics import accuracy_score as gpu_accuracy
acc_gpu = gpu_accuracy(yva, gclf.predict(Xva))
t_gpu = time.perf_counter() - t0
print(f"cuML  : akurasi={float(acc_gpu):.4f}  waktu={t_gpu:.2f}s")

In [ ]:
# Perbandingan apple-to-apple di CPU (data sama, tanpa duplikasi 20x)
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

train_p = pd.read_csv("smsa_train.tsv", sep="\t", names=["text", "label"])
valid_p = pd.read_csv("smsa_valid.tsv", sep="\t", names=["text", "label"])

t0 = time.perf_counter()
vec = TfidfVectorizer(max_features=20_000)
Xtr = vec.fit_transform(train_p.text)
Xva = vec.transform(valid_p.text)
clf = LogisticRegression(max_iter=200)
clf.fit(Xtr, train_p.label)
acc_cpu = accuracy_score(valid_p.label, clf.predict(Xva))
t_cpu = time.perf_counter() - t0
print(f"sklearn: akurasi={acc_cpu:.4f}  waktu={t_cpu:.2f}s")
ratio = t_cpu / t_gpu
if ratio >= 1:
    print(f"speedup GPU: {ratio:.1f}x")
else:
    print(f"GPU lebih lambat ({ratio:.1f}x) — cek: apakah sel warm-up di atas terlewati?")

### Native vs accel — kapan pakai yang mana?

| | `cuml.accel` (Babak 1) | API native (Babak 4) |
|---|---|---|
| Perubahan kode | nol | import diganti |
| Kontrol | terbatas (otomatis) | penuh (dtype, memori GPU) |
| Cocok untuk | migrasi cepat kode lama | pipeline GPU baru dari awal |

Dua pelajaran benchmark dari babak ini: **(1) Jangan ukur pemanggilan GPU pertama** — tanpa warm-up, cuML bisa tampak lebih lambat dari sklearn (±0,6x di T4) karena waktunya habis untuk inisialisasi CUDA, bukan komputasi; setelah panas, GPU menang dua digit bahkan di data sekecil SmSA. **(2) GPU menang di skala** — makin besar datanya, makin lebar jaraknya; itulah mengapa Babak 2 memakai 150 ribu paragraf.

## Babak 5 — Naik Kelas: NeMo Curator & Jembatan ke LLM 🌉

Semua yang kamu kerjakan hari ini adalah miniatur dari pipeline data LLM sungguhan:

| Hari ini (1× T4, 150 ribu paragraf) | Produksi (NeMo Curator, ratusan GPU, miliaran dokumen) |
|---|---|
| `tokenize`, `normalize` di nvtext | language ID, cleaning, filtering kualitas |
| MinHash + groupby (Babak 3) | fuzzy dedup MinHash-LSH terdistribusi |
| TF-IDF + LogReg (Babak 4) | classifier kualitas dokumen |

**Peta library NVIDIA untuk NLP** — yang sudah dan akan kamu temui:
- **RAPIDS (cuDF/nvtext/cuML)** — hari ini ✅
- **NeMo + NeMo Curator** — framework training LLM & kurasi data → **Day 06**
- **TensorRT-LLM, Triton** — inference cepat → **Day 06**
- **HF Transformers di GPU** — **Day 04**, besok kita masuk ke dunia LLM

### Latihan mandiri
1. Ganti korpus Babak 2 dengan 40 ribu artikel (`if i >= 40_000`). Apakah speedup GPU membesar atau mengecil? Mengapa?
2. Di Babak 3, perkecil `width` minhash menjadi 3. Apa efeknya terhadap jumlah kelompok duplikat, dan mengapa?
3. Di Babak 4, tambahkan `ngram_range=(1, 2)` pada kedua vectorizer. Bandingkan akurasi dan waktunya.